In [1]:
import rasterio
import numpy as np

def reclassify_chm(input_file, output_file):
    """
    Reclasifica Canopy Height Model según rangos especificados
    """
    with rasterio.open(input_file) as src:
        # Leer el array
        chm = src.read(1)
        
        # Crear array de salida
        reclassified = np.zeros_like(chm, dtype=np.int16)
        
        # Aplicar reclasificación
        reclassified[chm <= 0.1] = 0
        reclassified[(chm > 0.1) & (chm <= 0.3)] = 1
        reclassified[(chm > 0.3) & (chm <= 2)] = 2
        reclassified[chm > 2] = 3
        
        # Manejar NoData
        if src.nodata is not None:
            reclassified[chm == src.nodata] = -9999
        
        # Copiar metadatos y actualizar
        meta = src.meta.copy()
        meta.update({
            'dtype': 'int16',
            'nodata': -9999
        })
        
        # Guardar
        with rasterio.open(output_file, 'w', **meta) as dst:
            dst.write(reclassified, 1)

In [6]:
## 0 NoDatas handling

import rasterio
import numpy as np

def reclassify_chm_sin_nodata(input_file, output_file):
    with rasterio.open(input_file) as src:
        chm = src.read(1, masked=False)
        
        reclassified = np.zeros(chm.shape, dtype=np.int16)
        
        # Todo lo que sea NoData original o <= 0.1 va a clase 0
        mask_valid = chm != src.nodata if src.nodata else np.ones_like(chm, dtype=bool)
        
        reclassified[(chm > 0.1) & (chm <= 0.3) & mask_valid] = 1
        reclassified[(chm > 0.3) & (chm <= 2) & mask_valid] = 2
        reclassified[(chm > 2) & mask_valid] = 3
        # El resto (incluyendo NoData) queda en 0
        
        meta = src.meta.copy()
        meta.update({'dtype': 'int16', 'nodata': None})
        
        with rasterio.open(output_file, 'w', **meta) as dst:
            dst.write(reclassified, 1)

In [8]:
ch2014 = 'ruta/al/chm_2014.tif'
ch2014_reclass = 'ruta/de/salida/chm_2014_reclass.tif'

ch2020 = 'ruta/al/chm_2020.tif'
ch2020_reclass = 'ruta/de/salida/chm_2020_reclass.tif'

# Usar la función
reclassify_chm_sin_nodata(ch2014, ch2014_reclass)
#reclassify_chm('chm_input2.tif', 'chm_reclassified2.tif')

## Lacoon Stats 

In [13]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
import pandas as pd
import numpy as np

def calcular_superficie_por_clase(raster_file, lagunas_shp, fecha):
    """
    Calcula superficie (en m²) de cada clase CHM en cada laguna
    """
    # Leer lagunas
    lagunas = gpd.read_file(lagunas_shp)
    
    # Obtener resolución del pixel
    with rasterio.open(raster_file) as src:
        pixel_area = abs(src.transform[0] * src.transform[4])  # m² por pixel
    
    resultados = []
    
    for idx, laguna in lagunas.iterrows():
        # Calcular histograma de valores dentro de cada laguna
        stats = zonal_stats(
            laguna.geometry,
            raster_file,
            categorical=True,
            nodata=-9999,
            geojson_out=False
        )
        
        # Extraer conteos por clase
        conteos = stats[0] if stats else {}
        
        # Calcular superficies
        superficie_clase_0 = conteos.get(0, 0) * pixel_area
        superficie_clase_1 = conteos.get(1, 0) * pixel_area
        superficie_clase_2 = conteos.get(2, 0) * pixel_area
        superficie_clase_3 = conteos.get(3, 0) * pixel_area
        
        # Si querías incluir NoData en clase 0:
        # superficie_clase_0 += conteos.get(-9999, 0) * pixel_area
        
        resultados.append({
            'laguna_id': laguna['FORMACION'],
            'laguna_nombre': laguna.get('NOMBRE', laguna['FORMACION']),  # usa NOMBRE si existe, si no usa FORMACION
            'fecha': fecha,
            'superficie_clase_0_m2': superficie_clase_0,
            'superficie_clase_1_m2': superficie_clase_1,
            'superficie_clase_2_m2': superficie_clase_2,
            'superficie_clase_3_m2': superficie_clase_3,
            'superficie_total_m2': superficie_clase_0 + superficie_clase_1 + 
                                   superficie_clase_2 + superficie_clase_3
        })
    
    return pd.DataFrame(resultados)

In [17]:
## Calculamos (h29)

labordette = 'ruta/a/tu/inventario_lagunas.shp'
ch14 = 'ruta/de/salida/chm_2014_reclass.tif'
ch20 = 'ruta/de/salida/chm_2020_reclass.tif'
ch24 = 'ruta/al/chm_2024.tif'

# Procesar las 3 fechas
df_2014 = calcular_superficie_por_clase(ch14, labordette, 2014)
df_2020 = calcular_superficie_por_clase(ch20, labordette, 2020)
df_2024 = calcular_superficie_por_clase(ch24, labordette, 2024)

# Combinar todo
df_completo = pd.concat([df_2014, df_2020, df_2024], ignore_index=True)

# Calcular porcentajes
for clase in [0, 1, 2, 3]:
    df_completo[f'porcentaje_clase_{clase}'] = (
        df_completo[f'superficie_clase_{clase}_m2'] / 
        df_completo['superficie_total_m2'] * 100
    )

# Guardar resultados
df_completo.to_csv('analisis_matorrizacion_lagunas.csv', index=False)
df_completo.to_excel('analisis_matorrizacion_lagunas.xlsx', index=False)

print(df_completo)

            laguna_id             laguna_nombre  fecha  superficie_clase_0_m2  \
0       FORMACIÓN Nº1                EL LUCIO I   2014            5756.250000   
1       FORMACIÓN Nº2               EL LUCIO II   2014            2718.750000   
2       FORMACIÓN Nº3       LLANOS DE VELÁZQUEZ   2014           12762.500000   
3       FORMACIÓN Nº4                      None   2014            2381.250000   
4       FORMACIÓN Nº5                      None   2014            2693.750000   
...               ...                       ...    ...                    ...   
1768  FORMACIÓN Nº595                      None   2024             180.314683   
1769  FORMACIÓN Nº596   LAGUNA DE LA HIGUERUELA   2024            1887.359733   
1770  FORMACIÓN Nº597           LOS CHARQUILLOS   2024            7248.253967   
1771  FORMACIÓN Nº598  LAGUNA DE JUAN DE DIOS V   2024            1778.378331   
1772  FORMACIÓN Nº637                      None   2024            9592.344848   

      superficie_clase_1_m2

## Visualización: Evolución de la cobertura de vegetación

Comparamos la distribución de clases CHM en las lagunas para los tres años disponibles (2014, 2020, 2024) mediante:

1. **Stacked bars** — superficie total y porcentaje medio agregado por año
2. **Heatmap** — % de vegetación alta (clase 3) por laguna × año (top 40)
3. **Slope chart** — top 25 lagunas con mayor incremento de vegetación alta

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import numpy as np

# ── Paleta y etiquetas ──────────────────────────────────────────────────────
COLORES = {
    0: "#d4e6f1",  # sin vegetación
    1: "#a9dfbf",  # vegetación muy baja (0.1–0.3m)
    2: "#27ae60",  # matorral (0.3–2m)
    3: "#1a5276",  # arbolado / veg. alta (>2m)
}
ETIQUETAS = {
    0: "Sin vegetación (≤0.1m)",
    1: "Veg. muy baja (0.1–0.3m)",
    2: "Matorral (0.3–2m)",
    3: "Arbolado / veg. alta (>2m)",
}
AÑOS = [2014, 2020, 2024]

fig = plt.figure(figsize=(18, 14))
fig.suptitle(
    "Evolución de la cobertura de vegetación en las lagunas\n(PNOA LiDAR 2014–2020–2024)",
    fontsize=16, fontweight="bold", y=0.98,
)

# ── 1. Stacked bar — superficie total agregada por año ──────────────────────
ax1 = fig.add_subplot(2, 2, 1)

resumen = (
    df_completo.groupby("fecha")[[f"superficie_clase_{c}_m2" for c in range(4)]].sum()
    / 1e4  # → hectáreas
)

bottom = np.zeros(len(AÑOS))
for c in range(4):
    vals = [resumen.loc[a, f"superficie_clase_{c}_m2"] for a in AÑOS]
    ax1.bar(AÑOS, vals, bottom=bottom, color=COLORES[c], label=ETIQUETAS[c],
            width=2.5, edgecolor="white", linewidth=0.8)
    bottom += np.array(vals)

ax1.set_title("Superficie total por clase (todas las lagunas)", fontsize=11)
ax1.set_ylabel("Superficie (ha)")
ax1.set_xticks(AÑOS)
ax1.legend(loc="upper left", fontsize=8, framealpha=0.8)
ax1.grid(axis="y", alpha=0.3)
ax1.spines[["top", "right"]].set_visible(False)

# ── 2. Stacked bar — porcentaje medio por año ───────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)

resumen_pct = df_completo.groupby("fecha")[[f"porcentaje_clase_{c}" for c in range(4)]].mean()

bottom = np.zeros(len(AÑOS))
for c in range(4):
    vals = [resumen_pct.loc[a, f"porcentaje_clase_{c}"] for a in AÑOS]
    ax2.bar(AÑOS, vals, bottom=bottom, color=COLORES[c], label=ETIQUETAS[c],
            width=2.5, edgecolor="white", linewidth=0.8)
    bottom += np.array(vals)

ax2.set_title("Porcentaje medio por clase (todas las lagunas)", fontsize=11)
ax2.set_ylabel("% superficie media")
ax2.set_xticks(AÑOS)
ax2.set_ylim(0, 100)
ax2.grid(axis="y", alpha=0.3)
ax2.spines[["top", "right"]].set_visible(False)

# ── 3. Heatmap — % vegetación alta (clase 3) por laguna × año ───────────────
ax3 = fig.add_subplot(2, 1, 2)

pivot = df_completo.pivot_table(
    index="laguna_id", columns="fecha", values="porcentaje_clase_3", aggfunc="mean"
)

# Top 40 lagunas con mayor % veg. alta en 2024
top_lagunas = pivot[2024].nlargest(40).index
pivot_top = pivot.loc[top_lagunas].fillna(0)

# Etiquetas con nombre si existe
nombres = (
    df_completo[["laguna_id", "laguna_nombre"]]
    .drop_duplicates()
    .set_index("laguna_id")
)
etiq = [
    f"{lid} – {nombres.loc[lid, 'laguna_nombre']}"
    if lid in nombres.index and pd.notna(nombres.loc[lid, "laguna_nombre"])
    else lid
    for lid in pivot_top.index
]

sns.heatmap(
    pivot_top, ax=ax3, cmap="YlGn", linewidths=0.3, linecolor="white",
    annot=True, fmt=".0f", annot_kws={"size": 7},
    cbar_kws={"label": "% arbolado / veg. alta", "shrink": 0.6},
    vmin=0, vmax=100,
)
ax3.set_yticklabels(etiq, fontsize=7, rotation=0)
ax3.set_xticklabels(AÑOS, rotation=0)
ax3.set_title("Top 40 lagunas con mayor cobertura de vegetación alta en 2024 (%)", fontsize=11)
ax3.set_xlabel("")
ax3.set_ylabel("")

plt.tight_layout()
plt.savefig("evolucion_cobertura_lagunas.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: evolucion_cobertura_lagunas.png")


In [ ]:
# ── Slope chart: top 25 lagunas con mayor incremento de veg. alta 2014→2024 ──
fig, ax = plt.subplots(figsize=(10, 10))

pivot = df_completo.pivot_table(
    index="laguna_id", columns="fecha", values="porcentaje_clase_3", aggfunc="mean"
)
delta = (pivot[2024] - pivot[2014]).dropna().nlargest(25)
sel = pivot.loc[delta.index]

cmap = plt.cm.RdYlGn_r
norm = mcolors.Normalize(vmin=delta.min(), vmax=delta.max())

nombres = (
    df_completo[["laguna_id", "laguna_nombre"]]
    .drop_duplicates()
    .set_index("laguna_id")
)

for lid in sel.index:
    vals = sel.loc[lid, [2014, 2020, 2024]].values
    color = cmap(norm(delta[lid]))
    ax.plot(AÑOS, vals, "o-", color=color, lw=1.8, ms=6, alpha=0.9)
    nombre = (
        nombres.loc[lid, "laguna_nombre"]
        if lid in nombres.index and pd.notna(nombres.loc[lid, "laguna_nombre"])
        else lid
    )
    ax.text(2024.3, vals[-1], nombre, fontsize=7, va="center", color=color)
    ax.text(2013.7, vals[0], f"{vals[0]:.0f}%", fontsize=7, va="center",
            ha="right", color=color)

ax.set_xticks(AÑOS)
ax.set_xlim(2011, 2030)
ax.set_ylabel("% vegetación alta / arbolado (clase 3)")
ax.set_title(
    "Top 25 lagunas con mayor incremento de vegetación alta\n2014 → 2024",
    fontsize=12, fontweight="bold",
)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label="Incremento % (2014→2024)", shrink=0.4)
plt.tight_layout()
plt.show()
